# 🤖🔥 04_train_multitask_small — The Final Training Dojo ⚔️🐟💫

Welcome, TL-senpai.  
Your model is about to:
- classify species 🐠  
- estimate freshness 🍃🧪  
- predict head/tail keypoints 🎯  

We train in **3 deadly stages**, like boss phases in an JRPG:
1️⃣ Species head only  
2️⃣ Freshness + keypoints  
3️⃣ Fine-tuning the backbone (careful… it’s spicy)  

Let’s forge this warrior model. ⚡🔥


## Cell 1 — Imports, hyperparams, load metadata (Code)

In [10]:
df_sp = df[df['source']=='species']
df_ds = df[df['source']=='disease']
print(len(df_sp), len(df_ds))

# Also check if resized images exist for species/disease
import os
df_sp['exists'] = df_sp['image_224'].apply(lambda p: os.path.exists(p))
df_ds['exists'] = df_ds['image_224'].apply(lambda p: os.path.exists(p))

print("Species valid images:", df_sp['exists'].sum())
print("Disease valid images:", df_ds['exists'].sum())


9000 2444
Species valid images: 9000
Disease valid images: 2444


C:\Users\Hp\AppData\Local\Temp\ipykernel_8092\1223951190.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sp['exists'] = df_sp['image_224'].apply(lambda p: os.path.exists(p))
C:\Users\Hp\AppData\Local\Temp\ipykernel_8092\1223951190.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ds['exists'] = df_ds['image_224'].apply(lambda p: os.path.exists(p))


## 🧪 Cell 1 — Imports, load metadata, setup dojo

In [11]:
# Cell 1
from pathlib import Path
import pandas as pd, numpy as np, tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

PROC = Path("data_sih/processed")
META = PROC/"metadata_final_precomputed.csv"
if not META.exists():
    META = PROC/"metadata_with_kp.csv" if (PROC/"metadata_with_kp.csv").exists() else PROC/"metadata_rebuilt_mapped.csv"
df = pd.read_csv(META)
print("Using metadata:", META.name, "| rows:", len(df))

IMG_SIZE = 224
BATCH = 16
AUTOTUNE = tf.data.AUTOTUNE
NUM_EPOCHS_STAGE1 = 4
NUM_EPOCHS_STAGE2 = 6
NUM_EPOCHS_STAGE3 = 4


Using metadata: metadata_final_precomputed.csv | rows: 12064


## 🗂️ Cell 2 — species mapping + keypoint normalization

In [12]:
# Cell 2
df['species'] = df['species'].fillna("unknown")
species_list = sorted(df['species'].unique())
species_map = {s:i for i,s in enumerate(species_list)}
NUM_SPECIES = len(species_list)
print("NUM_SPECIES:", NUM_SPECIES)

kp_cols = ['head_x','head_y','tail_x','tail_y']
# ensure kp columns exist
for c in kp_cols:
    if c not in df.columns:
        df[c] = np.nan

# If KP values are in pixel coords (likely), normalize to [0,1] relative to IMG_SIZE
# detect if any value > 1.5 => pixel coords
for c in kp_cols:
    if df[c].dropna().shape[0] and df[c].dropna().max() > 1.5:
        df[c] = df[c] / IMG_SIZE

# replace missing with sentinel -1
for c in kp_cols:
    df[c] = df[c].fillna(-1.0)
df['freshness'] = df['freshness'].fillna(-1.0)

# add species_id column for convenience
df['species_id'] = df['species'].map(species_map)


NUM_SPECIES: 10


## 📡 Cell 3 — Build tf.data pipeline from resized images

In [13]:
# Cell 3
def make_dataset(df_in, batch=BATCH, shuffle=True, augment=False):
    df_local = df_in.reset_index(drop=True).copy()
    image_paths = df_local['image_224'].astype(str).tolist()
    species_ids = df_local['species_id'].astype(np.int32).tolist()
    freshness_vals = df_local['freshness'].astype(np.float32).tolist()
    kps = df_local[kp_cols].astype(np.float32).values.tolist()

    import tensorflow as tf
    def gen():
        for p, sp, fr, kp in zip(image_paths, species_ids, freshness_vals, kps):
            yield p.encode('utf-8'), np.int32(sp), np.float32(fr), np.array(kp, dtype=np.float32)

    output_signature = (
        tf.TensorSpec(shape=(), dtype=tf.string),
        tf.TensorSpec(shape=(), dtype=tf.int32),
        tf.TensorSpec(shape=(), dtype=tf.float32),
        tf.TensorSpec(shape=(4,), dtype=tf.float32)
    )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    def _parse(path, spid, fr, kp):
        img = tf.io.read_file(tf.cast(path, tf.string))
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.08)
            img = tf.image.random_contrast(img, 0.9, 1.1)
        sp = tf.one_hot(spid, NUM_SPECIES)
        return img, {"species_out": sp, "freshness_out": fr, "kp_out": kp}

    ds = ds.map(_parse, num_parallel_calls=AUTOTUNE)
    if shuffle: ds = ds.shuffle(1024)
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

train_df = df.sample(frac=0.88, random_state=42).reset_index(drop=True)
val_df = df.drop(train_df.index).reset_index(drop=True)
train_ds = make_dataset(train_df, augment=True)
val_ds = make_dataset(val_df, shuffle=False)
print("Train rows:", len(train_df), "Val rows:", len(val_df))


Train rows: 10616 Val rows: 1448


## 🧱 Cell 4 — Build MobileNetV2 Multi-task Model

In [14]:
# Cell 4
from tensorflow.keras import layers
def build_multitask_model(img_size=IMG_SIZE, num_species=NUM_SPECIES):
    inp = layers.Input(shape=(img_size,img_size,3), name="image_input")
    base = tf.keras.applications.MobileNetV2(include_top=False, input_tensor=inp, weights='imagenet', pooling='avg')
    feat = base.output

    sp = layers.Dense(512, activation='relu')(feat)
    sp = layers.Dropout(0.3)(sp)
    species_out = layers.Dense(num_species, activation='softmax', name='species_out')(sp)

    fr = layers.Dense(256, activation='relu')(feat)
    fr = layers.Dropout(0.3)(fr)
    freshness_out = layers.Dense(1, activation='sigmoid', name='freshness_out')(fr)

    kp = layers.Dense(256, activation='relu')(feat)
    kp = layers.Dropout(0.3)(kp)
    kp_out = layers.Dense(4, activation='linear', name='kp_out')(kp)

    return Model(inputs=inp, outputs=[species_out, freshness_out, kp_out])

model = build_multitask_model()
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 112, 112, 32  864         ['image_input[0][0]']            
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 112, 112, 32  128         ['Conv1[0][0]']                  
                                )                                                             

## ⚙️ Cell 5 — Compile with masked loss

In [15]:
# Cell 5
import tensorflow.keras.backend as K

def masked_mse(y_true, y_pred):
    mask = K.cast(K.not_equal(y_true, -1.0), K.floatx())
    mask = K.reshape(mask, (-1,))
    y_true = K.reshape(y_true, (-1,))
    y_pred = K.reshape(y_pred, (-1,))
    denom = K.maximum(K.sum(mask), 1.0)
    return K.sum(mask * K.square(y_true - y_pred)) / denom

losses = {"species_out":"categorical_crossentropy", "freshness_out":masked_mse, "kp_out":"mse"}
loss_weights = {"species_out":1.0, "freshness_out":1.0, "kp_out":0.5}
opt = tf.keras.optimizers.Adam(1e-4)

model.compile(optimizer=opt, loss=losses, loss_weights=loss_weights,
              metrics={"species_out":"accuracy", "freshness_out":tf.keras.metrics.RootMeanSquaredError(), "kp_out":tf.keras.metrics.RootMeanSquaredError()})


## 🧿 Cell 6 — Callbacks (checkpoints + LR scheduler)

In [16]:
# Cell 6
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
MODEL_DIR = PROC/"models"; MODEL_DIR.mkdir(exist_ok=True)
cp_path = MODEL_DIR/"sih_multitask_best.h5"
checkpoint = ModelCheckpoint(str(cp_path), monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
callbacks = [checkpoint, reduce_lr, early]


## 🥋 Cell 7 — Stage 1: species head only (backbone frozen)

In [ ]:
# Cell 7 — Stage 1: train species head only
for layer in model.layers:
    if layer.name.startswith("mobilenetv2"):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=losses, loss_weights=loss_weights, metrics={"species_out":"accuracy"})
history1 = model.fit(train_ds, validation_data=val_ds, epochs=NUM_EPOCHS_STAGE1, callbacks=callbacks)


Epoch 1/4


## 🧨 Cell 8 — Stage 2: freshness + keypoints

In [ ]:
# Cell 8 — Stage 2: train freshness + kp heads (backbone frozen)
for layer in model.layers:
    if layer.name.startswith("mobilenetv2"):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=losses, loss_weights=loss_weights,
              metrics={"species_out":"accuracy","freshness_out":tf.keras.metrics.RootMeanSquaredError(),"kp_out":tf.keras.metrics.RootMeanSquaredError()})
history2 = model.fit(train_ds, validation_data=val_ds, epochs=NUM_EPOCHS_STAGE2, callbacks=callbacks)


## 🔥 Cell 9 — Stage 3: fine-tune last N backbone layers

In [ ]:
# Cell 9 — Stage 3: fine-tune top backbone layers
N = 60
base_layers = [l for l in model.layers if l.name.startswith("mobilenetv2")]
for l in base_layers: l.trainable = False
for l in base_layers[-N:]: l.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=losses, loss_weights=loss_weights,
              metrics={"species_out":"accuracy","freshness_out":tf.keras.metrics.RootMeanSquaredError(),"kp_out":tf.keras.metrics.RootMeanSquaredError()})
history3 = model.fit(train_ds, validation_data=val_ds, epochs=NUM_EPOCHS_STAGE3, callbacks=callbacks)


## 💾 Cell 10 — Save final model + sanity check

In [ ]:
# Cell 10
final_path = MODEL_DIR/"sih_multitask_final.h5"
model.save(final_path)
print("Saved model:", final_path)
for batch in val_ds.take(1):
    imgs, _ = batch
    preds = model.predict(imgs)
    print("Species probs shape:", preds[0].shape)
    print("Freshness sample:", preds[1].flatten()[:5])
    print("KP sample:", preds[2][0])
    break


## 👁️ Cell 11 — Visualize predictions

In [ ]:
# Cell 11
import numpy as np
def visualize_preds(ds, model, n=6):
    for imgs, targets in ds.take(1):
        preds = model.predict(imgs)
        sp_probs, fr_vals, kp_vals = preds
        imgs_np = imgs.numpy()
        for i in range(min(n, imgs_np.shape[0])):
            plt.figure(figsize=(4,4))
            plt.imshow((imgs_np[i]*255).astype('uint8'))
            sp_idx = np.argmax(sp_probs[i])
            plt.title(f"{species_list[sp_idx]} | freshness={fr_vals[i][0]:.2f}")
            hx = int(kp_vals[i][0]*IMG_SIZE); hy = int(kp_vals[i][1]*IMG_SIZE)
            tx = int(kp_vals[i][2]*IMG_SIZE); ty = int(kp_vals[i][3]*IMG_SIZE)
            if all(v>=0 for v in [hx,hy,tx,ty]):
                plt.scatter([hx,tx],[hy,ty], c=['r','y'])
            plt.axis('off')
            plt.show()

visualize_preds(val_ds, model, n=6)
